# B2-024-gpu-scientific-ml-capstone — Practice p18 — Solution

**Type:** integrative · **Difficulty:** advanced · **Concepts:** semi-supervised-pseudo-labeling

*65 minutes.*  
**Set:** C  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261022`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

## Supplied cell (copied verbatim from the statement)

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261022
LOADER_CANDIDATES = (
    Path("../data/capstone_data.py"),
    Path("units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
    Path("book2/units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("capstone_data", loader_path)
capstone_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(capstone_data)
torch.set_num_threads(1)

K = 3
LLOYD_ITERATIONS = 20
COMPARISON_THRESHOLD = 0.9
COMPARISON_CAP = 200
MARGIN = 0.03

## Solution

**Frozen model and features.** The labelled-only CNN is trained with exactly the committed recipe and frozen. `features(x)` returns the first convolution's pre-ReLU responses (8 filters × 6 × 6 positions = 288 numbers per image) under `torch.no_grad()`; while `fitting` is true it also hashes each input row (the seam).

**k-means.** Lloyd's algorithm with $k=3$ on the 630 rows $Z$ (labelled first), centroids initialized at the three labelled class means, exactly 20 iterations of assign (`torch.cdist` + `argmin`, which returns the lowest index on ties) then update (mean of assigned rows; an empty cluster's centroid stays put).

**Naming and prediction.** Each cluster is named by the majority label of the labelled training images assigned to it (ties to the lowest class), and a cluster without labelled members takes the label of the nearest labelled-class centroid. A new image's class is the name of its nearest final centroid.

In [ ]:
xl, yl = capstone_data.train_rows("shapes_ssl")
xv, yv = capstone_data.val_rows("shapes_ssl")
pool = capstone_data.unlabelled_rows()

# Frozen labelled-only CNN (committed recipe)
torch.manual_seed(SEED)
model = capstone_data.ShapeCNN()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)
for _ in range(150):
    optimizer.zero_grad(set_to_none=True)
    loss = F.cross_entropy(model(xl), yl)
    loss.backward()
    optimizer.step()
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

fitting = False
SEAM = []


def features(x):
    if fitting:
        SEAM.extend(capstone_data.split_of("shapes_ssl", row) for row in x)
    with torch.no_grad():
        return model.conv1(x).flatten(1)


fitting = True
Z = torch.cat([features(xl), features(pool)])
fitting = False
F_l = Z[: xl.shape[0]]

labelled_centroids = torch.stack([F_l[yl == j].mean(dim=0) for j in range(K)])
centroids = labelled_centroids.clone()
iterations = 0
for _ in range(LLOYD_ITERATIONS):
    assign = torch.cdist(Z, centroids).argmin(dim=1)
    new = centroids.clone()
    for j in range(K):
        members = assign == j
        if members.any():
            new[j] = Z[members].mean(dim=0)
    centroids = new
    iterations += 1

# Naming
lab_assign = torch.cdist(F_l, centroids).argmin(dim=1)
names = []
for j in range(K):
    member_labels = yl[lab_assign == j]
    if len(member_labels) > 0:
        names.append(int(torch.bincount(member_labels, minlength=K).argmax()))     # argmax: lowest class on ties
    else:
        names.append(int(torch.cdist(centroids[j:j + 1], labelled_centroids).argmin()))
NAMES = torch.tensor(names, dtype=torch.int64)


def ctl_predict(images):
    return NAMES[torch.cdist(features(images), centroids).argmin(dim=1)]


cluster_sizes = torch.bincount(torch.cdist(Z, centroids).argmin(dim=1), minlength=K).tolist()
print("Z", tuple(Z.shape), "| iterations", iterations, "| cluster sizes", cluster_sizes, "| names", names)

**Comparison with threshold pseudo-labels (validation and pool only).**
The frozen model's softmax on the pool gives threshold pseudo-labels ($\tau=0.9$, cap 200, p09's rule). Agreement is measured on those selected pool images (no pool labels are needed); accuracies are measured on the labelled validation images. The one-round noisy student is p17's `train_student` with noise $0.3$ on `xl` plus these pseudo-labels.

In [ ]:
def select_pseudo_labels(probs, threshold, max_per_class):
    pred = probs.argmax(dim=1)
    conf = probs.max(dim=1).values
    candidate = conf >= threshold
    kept = []
    for k in range(probs.shape[1]):
        rows = torch.nonzero(candidate & (pred == k)).flatten()
        rows = rows[torch.sort(-conf[rows], stable=True).indices]
        kept.append(rows[:max_per_class])
    indices = torch.sort(torch.cat(kept)).values.to(torch.int64)
    return indices, pred[indices].to(torch.int64)


class NoisyStudent(nn.Module):
    def __init__(self, noise_std):
        super().__init__()
        self.noise_std = noise_std
        self.cnn = capstone_data.ShapeCNN()

    def forward(self, x):
        if self.training and self.noise_std > 0:
            x = x + self.noise_std * torch.randn_like(x)
        return self.cnn(x)


def train_student(x, y, noise_std):
    torch.manual_seed(SEED)
    student = NoisyStudent(noise_std)
    opt = torch.optim.Adam(student.parameters(), lr=0.01, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)
    student.train()
    for _ in range(150):
        opt.zero_grad(set_to_none=True)
        F.cross_entropy(student(x), y).backward()
        opt.step()
    return student.eval()


with torch.no_grad():
    pool_probs = F.softmax(model(pool), dim=1)
pl_idx, pl_labels = select_pseudo_labels(pool_probs, COMPARISON_THRESHOLD, COMPARISON_CAP)
ctl_on_selected = ctl_predict(pool[pl_idx])
agreement = (ctl_on_selected == pl_labels).double().mean().item()
student = train_student(torch.cat([xl, pool[pl_idx]]), torch.cat([yl, pl_labels]), 0.3)

with torch.no_grad():
    VAL = {"frozen baseline": (model(xv).argmax(dim=1) == yv).double().mean().item(),
           "cluster-then-label": (ctl_predict(xv) == yv).double().mean().item(),
           "one-round noisy student": (student(xv).argmax(dim=1) == yv).double().mean().item()}
print(f"threshold pseudo-labels: {len(pl_idx)} selected, class counts {torch.bincount(pl_labels, minlength=3).tolist()}")
print(f"agreement of cluster-then-label with them: {agreement:.4f}")
for name, acc in VAL.items():
    print(f"validation accuracy, {name}: {acc:.4f}")
print("seam values:", sorted(set(SEAM)), "| rows hashed:", len(SEAM))

**Locked test score of cluster-then-label (one call, after all fitting).**

In [ ]:
baseline = capstone_data.baseline_score("shapes_ssl")


def predict_fn(images):
    return ctl_predict(images).to(torch.int64)


assert capstone_data.test_call_count("shapes_ssl") == 0
test = capstone_data.final_test_score(predict_fn, n_boot=1000, alpha=0.05, task="shapes_ssl")
assert capstone_data.test_call_count("shapes_ssl") == 1
print(f"cluster-then-label: test accuracy {test.score:.4f}, 95% CI [{test.ci_low:.4f}, {test.ci_high:.4f}]")
print(f"committed baseline: {baseline.score:.4f}; bar {baseline.score + MARGIN:.4f}")

**Why the frozen features decide the outcome.**
k-means never sees a label: it can only find the cloud structure that the frozen first-layer filters create, and the 30 labelled images merely name the clouds (here cluster $j$ is named class $j$ for $j=0,1,2$), so if the features put nuisance factors such as position, stroke width or contrast ahead of shape, the clusters follow the nuisance and no naming rule can repair it.
At this seed the geometry is good: on the 521 pool images the frozen CNN labels with confidence $\ge0.9$, cluster-then-label agrees with the threshold pseudo-label on $86.4\%$, so centroids fitted to 630 images and a linear head fitted to 30 carve the space largely the same way, and the centroids, averaged over hundreds of images, place the boundaries better (validation $0.733$ versus the frozen baseline's $0.600$, and a test accuracy above the bar).
The remaining $13.6\%$ disagreement marks images where the clouds overlap in this feature space, and with another initialization seed the filters, and hence that overlap, change: if a class's images split across clusters or two classes merge, agreement falls and the clusters can be worse than the baseline's head, which is why the method must be compared on validation.

### Answer check

In [ ]:
assert Z.shape == (630, 288)
assert iterations == LLOYD_ITERATIONS == 20
assert names == [0, 1, 2]                       # the three cluster names at the frozen seed
assert 0.0 <= agreement <= 1.0
assert len(SEAM) == 630 and set(SEAM) <= {"train", "unlabelled"}
assert capstone_data.test_call_count("shapes_ssl") == 1
assert test.score >= baseline.score + MARGIN